# Individual Income Tax Overview, Dependents, and Filing Status

**Selected Excercises**

McGraw Hill's Taxation of Individuals and Business Entities 2025

Spilker, Ayers, Lewis, Weaver, Barrick, Robinson, Worsham

In [48]:
import os
import sys
import pandas as pd
from IPython.display import display, HTML

mod_path = os.path.abspath(os.path.join(".."))
if mod_path not in sys.path:
    sys.path.append(mod_path)

from stackCalc import RPNEngine
from income_tax import OrdinaryIncomeTax, PreferentialIncomeTax

In [2]:
# Lets define some functions so we can reuse them later
def income_tax_presentation(gi, adj, fad, ti, cg, otl, cgt, crts, pre):
    """
    gi:   Gross income
    adj:  Adjustments to arrive at agi
    itm:  Itemized deductions
    fad:  From agi deductions - adjustments from agi. Greater of standard deduction or itemized. Computed by OrdinaryIncomeTax)
    qbi:  Qualified business income deduction
    ti:   Taxable income
    cg:   Capital gains
    otl:  Ordinary tax liability
    cgt:  Capital gains tax
    crts: Tax Credits 
    pre:  Prepayments of income tax
    """
    
    gross_income = gi + cg
    adjusted_gross_income = gross_income - adj
    taxable_income = adjusted_gross_income - fad
    taxable_ordinary_income = taxable_income - cg
    total_tax = otl + cgt
    tax_due_refund = total_tax + crts + pre
    results = "Tax due" if tax_due_refund >= 0 else "Tax refund"

    df = pd.DataFrame([
            ["Gross income", gross_income],
            ["Adjustments", adj],
            [""],
            ["Adjusted gross income", adjusted_gross_income],
            ["From AGI deductions", fad],
            [""],
            ["Taxable income", taxable_income],
            [""],
            ["Taxable ordinary income", taxable_ordinary_income],
            ["Taxable capital gains", cg],
            [""],
            ["Ordinary income tax", otl],
            ["Capital gains tax", cgt],
            [""],
            ["Tax before credits", total_tax],
            [""],
            ["Credits", crts],
            ["Prepayments", pre],
            [""],
            [results, tax_due_refund],
        ], columns=["", ""])

    def format_number(x):
        if pd.isna(x):
            return ""
        if x < 0:
            return f"({abs(x):,.0f})"
        return f"{x:,.0f}"
    
    amount_col = df.iloc[:, 1].astype(object).apply(format_number)
    df.isetitem(1, amount_col) # type: ignore

    return df

---

Jeremy (unmarried) earned 100,000 in salary and 6,000 in interest income during the year. Jeremy’s employer withheld 10,000 of federal income taxes from Jeremy’s paychecks during the year. Jeremy has one qualifying dependent child (age 14) who lives with him. Jeremy qualifies to file as head of household and has 25,000 in itemized deductions.

In [ ]:
status = "HOH"               # Head of Household
age = 0                      # Not applicable
salary = 100000              # Ordinary income
interest = 6000              # Ordinary income
adjustments = 0              # Above the line adjustments
itemized_deduction = 25000   # Below the line adjustments
standard_deduction = 24150   # Below the line adjustments - 2026 Standard Deduction
tax_withholdings = 10000     # Tax Prepayments
child_tax_credit = 2200      # CTC - Child Tax Credit 2026


**- a) Determine Jeremy’s tax refund or taxes due.**

In [5]:
gross_income = salary + interest
it = OrdinaryIncomeTax(status, age, gross_income, adjustments, itemized_deduction).calculate_tax()
capital_gains = 0
capital_gains_tax = 0
tax = income_tax_presentation(gross_income,
                              it["adjustments"],  
                              it["deduction"], 
                              it["taxable_income"],
                              capital_gains,
                              it["ordinary_tax"],
                              capital_gains_tax,
                              -child_tax_credit,
                              -tax_withholdings)

display(HTML(tax.to_html(index=False))) 

,
Gross income,"106,000"
Adjustments,0
,
Adjusted gross income,"106,000"
From AGI deductions,"25,000"
,
Taxable income,"81,000"
,
Taxable ordinary income,"81,000"
Taxable capital gains,0


**- b) Assume that in addition to the original facts, Jeremy has a long-term capital gain of 4,000. What is Jeremy’s tax refund or tax due including the tax on the capital gain?**

In [6]:
capital_gains = 4000
it = OrdinaryIncomeTax(status, age, gross_income, adjustments, itemized_deduction).calculate_tax()
cgt = PreferentialIncomeTax(status, age, gross_income, 0, capital_gains).calculate_tax()
capital_gains_tax = cgt[1]

tax = income_tax_presentation(gross_income,
                              it["adjustments"],  
                              it["deduction"], 
                              it["taxable_income"],
                              capital_gains,
                              it["ordinary_tax"],
                              capital_gains_tax,
                              -child_tax_credit,
                              -tax_withholdings)

display(HTML(tax.to_html(index=False))) 

,
Gross income,"110,000"
Adjustments,0
,
Adjusted gross income,"110,000"
From AGI deductions,"25,000"
,
Taxable income,"85,000"
,
Taxable ordinary income,"81,000"
Taxable capital gains,"4,000"



- c) Assume the original facts except that Jeremy has only 7,000 in itemized deductions. What is Jeremy’s tax refund or tax due?

In [7]:
itemized_deduction = 7000
it = OrdinaryIncomeTax(status, age, gross_income, adjustments, itemized_deduction).calculate_tax()
capital_gains = 0
capital_gains_tax = 0
tax = income_tax_presentation(gross_income,
                              it["adjustments"], 
                              it["deduction"], 
                              it["taxable_income"],
                              capital_gains,
                              it["ordinary_tax"],
                              capital_gains_tax,
                              -child_tax_credit,
                              -tax_withholdings)

display(HTML(tax.to_html(index=False))) 

,
Gross income,"106,000"
Adjustments,0
,
Adjusted gross income,"106,000"
From AGI deductions,"24,150"
,
Taxable income,"81,850"
,
Taxable ordinary income,"81,850"
Taxable capital gains,0


---

Aram’s taxable income before considering capital gains and losses is 60,000. Determine Aram’s taxable income and how much of the income will be taxed at ordinary rates in each of the following alternative scenarios (assume Aram files as a single taxpayer).

- a) Aram sold a capital asset that he owned for more than one year for a 5,000 gain, a capital asset that he owned for more than one year for a 500 loss, a capital asset that he owned for six months for a 1,200 gain, and a capital asset he owned for two months for a 900 loss.

In [5]:
status = "SNG"
age = 0                      # Not applicable
income = 60000               # Ordinary income
adjustments = 0              # Above the line adjustments
tax_withholdings = 0         # Tax Prepayments
child_tax_credit = 0         # CTC - Child Tax Credit 2026

lt_cap_gains = 5000
lt_cap_losses = 500
st_cap_gains = 1200
st_cap_losses = 900

# Netting process
lt_net = lt_cap_gains - lt_cap_losses
st_net = st_cap_gains - st_cap_losses

print(f"""
Aram's taxable income:

Taxable income:            {income:,.0f}
Short-term capital gains   {st_net:,.0f}
Long-term capital gains    {lt_net:,.0f}
                           ------
Taxable income             {income + st_net + lt_net:,.0f}

Aram's taxable income that will be taxed at ordinary rates:
      
Taxable income:            {income:,.0f}
Short-term capital gains   {st_net:,.0f}
                           ------
Ordinary income            {income + st_net:,.0f}
    """)


Aram's taxable income:

Taxable income:            60,000
Short-term capital gains   300
Long-term capital gains    4,500
                           ------
Taxable income             64,800

Aram's taxable income that will be taxed at ordinary rates:

Taxable income:            60,000
Short-term capital gains   300
                           ------
Ordinary income            60,300
    


- b) Aram sold a capital asset that he owned for more than one year for a 2,000 gain, a capital asset that he owned for more than one year for a 2,500 loss, a capital asset that he owned for six months for a 200 gain, and a capital asset he owned for two months for a 1,900 loss.

In [14]:
lt_cap_gains = 2000
lt_cap_losses = 2500
st_cap_gains = 200
st_cap_losses = 1900

# Netting process
lt_net = lt_cap_gains - lt_cap_losses
st_net = st_cap_gains - st_cap_losses

capital_loss_limit = min(abs(lt_net) + abs(st_net), 3000)


print(f"""
Netting Process:
Net long-term capital loss    {abs(lt_net):,.0f}
Net short-term capital loss   {abs(st_net):,.0f}
                              ------
Total capital loss            {abs(lt_net) + abs(st_net):,.0f}""")

print(f"""
Aram's taxable income:

Taxable income:               {income:,.0f}
Capital loss                  {capital_loss_limit:,.0f}
                              ------
Taxable income                {income - capital_loss_limit:,.0f}

Note: In this case the whole taxable income will be taxed at ordinary rates.""")



Netting Process:
Net long-term capital loss    500
Net short-term capital loss   1,700
                              ------
Total capital loss            2,200

Aram's taxable income:

Taxable income:               60,000
Capital loss                  2,200
                              ------
Taxable income                57,800

Note: In this case the whole taxable income will be taxed at ordinary rates.


c) Aram sold a capital asset that he owned for more than one year for a 2,500 loss, a capital asset that he owned for six months for a 4,200 gain, and a capital asset he owned for two months for a 300 loss.

In [22]:
lt_cap_gains = 0
lt_cap_losses = 2500
st_cap_gains = 4200
st_cap_losses = 300

# Netting process
lt_net = lt_cap_gains - lt_cap_losses
st_net = st_cap_gains - st_cap_losses
net_capital_gain = lt_net + st_net

print(f"""
Netting Process:
Net long-term capital loss            {lt_net:,.0f}
Net short-term capital gain            {st_net:,.0f}
                                      ------
Net short-term capital gain            {net_capital_gain:,.0f}""")

print(f"""
Aram's taxable income:

Taxable income:                       {income:,.0f}
Net short-term capital gain           {net_capital_gain:,.0f}
                                      ------
Taxable income                        {income + net_capital_gain:,.0f}

Note: In this case the whole taxable income will be taxed at ordinary rates.""")


Netting Process:
Net long-term capital loss            -2,500
Net short-term capital gain            3,900
                                      ------
Net short-term capital gain            1,400

Aram's taxable income:

Taxable income:                       60,000
Net short-term capital gain           1,400
                                      ------
Taxable income                        61,400

Note: In this case the whole taxable income will be taxed at ordinary rates.


- d) Aram sold a capital asset that he owned for more than one year for a 3,000 gain, a capital asset that he owned for more than one year for a 300 loss, a capital asset that he owned for six months for a 200 gain, and a capital asset he owned for two months for a 1,900 loss.

In [33]:
lt_cap_gains = 3000
lt_cap_losses = 300
st_cap_gains = 200
st_cap_losses = 1900

# Netting process
lt_net = lt_cap_gains - lt_cap_losses
st_net = st_cap_gains - st_cap_losses
net_capital_gain = lt_net + st_net

print(f"""
Netting Process:
Net long-term capital loss            {lt_net:,.0f}
Net short-term capital gain          {st_net:,.0f}
                                      ------
Net short-term capital gain           {net_capital_gain:,.0f}

Aram's taxable income:

Taxable income:                     {income:,.0f}
Net Long-term capital gains          {net_capital_gain:,.0f}
                                    ------
Taxable income                      {income + net_capital_gain:,.0f}

Aram's taxable income that will be taxed at ordinary rates:
      
Taxable income:                     {income:,.0f}
                                    ------
Ordinary income                     {income:,.0f}
    """)



Netting Process:
Net long-term capital loss            2,700
Net short-term capital gain          -1,700
                                      ------
Net short-term capital gain           1,000

Aram's taxable income:

Taxable income:                     60,000
Net Long-term capital gains          1,000
                                    ------
Taxable income                      61,000

Aram's taxable income that will be taxed at ordinary rates:

Taxable income:                     60,000
                                    ------
Ordinary income                     60,000
    


---

David and Lilly Fernandez have determined their tax liability on their joint tax return to be 3,100. They have made prepayments of 1,900 and also have a child tax credit of 2,000. What is the amount of their tax refund or taxes due?

In [45]:
tax_liability = 3100
prepayments = 1900
child_tax_credit = 2000
tax_due_refund = tax_liability - prepayments - child_tax_credit
due_or_refund = "Tax refund" if tax_due_refund < 0 else "Tax due"
print(f"""
Tax liability       {tax_liability:,.0f}
Prepayments         {prepayments:,.0f}
Child Tax Credit    {child_tax_credit:,.0f}
                    -----
{due_or_refund}           {abs(tax_due_refund)}""")



Tax liability       3,100
Prepayments         1,900
Child Tax Credit    2,000
                    -----
Tax refund           800


---

Ekiya, who is single, has been offered a position as a city landscape consultant. The position pays 125,000 in wages. Assume Ekiya has no dependents. Ekiya deducts the standard deduction instead of itemized deductions, and she is not eligible for the qualified business income deduction.

- a) What is the amount of Ekiya’s after-tax compensation (ignore payroll taxes)?

In [58]:
status = "SNG"               # Head of Household
age = 0                      # Not applicable
job1_wages = 125000               # Ordinary income
adjustments = 0              # Above the line adjustments
itemized_deduction = 0   # Below the line adjustments

job1_tax = OrdinaryIncomeTax(status, age, job1_wages, adjustments, itemized_deduction).calculate_tax()["ordinary_tax"]
job1_after_tax_compensation = job1_wages - tax

print(f"""
Wages                     {job1_wages:,.0f}
Tax                        {job1_tax:,.0f}
                          --------
After-tax Compensation    {job1_after_tax_compensation:,.0f}

Note: Standard deduction for a single tax payer in 2026 is: $16,100
      """)



Wages                     125,000
Tax                        18,734
                          --------
After-tax Compensation    106,266

Note: Standard deduction for a single tax payer in 2026 is: $16,100
      


b) Suppose Ekiya receives a competing job offer of $120,000 in wages and nontaxable
(excluded) benefits worth $5,000. What is the amount of Ekiya’s after-tax compensation for
the competing offer? Which job should she take if taxes are the only concern?

In [64]:
job2_wages = 120000
non_taxable_benefits = 5000
compensation = job2_wages + non_taxable_benefits
job2_tax = OrdinaryIncomeTax(status, age, job2_wages, adjustments, itemized_deduction).calculate_tax()["ordinary_tax"]
job2_after_tax_compensation = compensation - job2_tax

comparison = {
    "": ["Wages", "Non Taxable Compensation", "Tax", "After Tax Compensation"],
    "Job 1": [job1_wages, 0, job1_tax, job1_after_tax_compensation],
    "Job 2": [job2_wages, non_taxable_benefits, job2_tax, job2_after_tax_compensation],
}

df = pd.DataFrame(comparison)
df["Job 1"] = df["Job 1"].map("${:,.0f}".format)
df["Job 2"] = df["Job 2"].map("${:,.0f}".format)
display(HTML(df.to_html(index=False)))

print(f"""
Ekiya should choose Job 2 which offers the best after tax compensation along with additional benefits.

Note: Standard deduction for a single tax payer in 2026 is: $16,100
""")

,Job 1,Job 2
Wages,"$125,000","$120,000"
Non Taxable Compensation,$0,"$5,000"
Tax,"$18,734","$17,570"
After Tax Compensation,"$106,266","$107,430"



Ekiya should choose Job 2 which offers the best after tax compensation along with additional benefits.

Note: Standard deduction for a single tax payer in 2026 is: $16,100

